In [1]:
# --- fact_weather ---
# CREATE TABLE fact_weather(
#     weather_id serial primary key,
#     session_id int not null,
#     timestamp_offset_s int not null,
#     air_temp_c numeric not null,
#     track_temp_c numeric not null,
#     humidity_pct numeric not null,
#     rainfall boolean not null,
#     wind_direction int not null,
#     wind_speed_ms numeric not null,
#     pressure_mbar numeric not null,
#     CONSTRAINT fk_session_weather FOREIGN KEY (session_id)
#     REFERENCES dim_session(session_id)
#     ON DELETE RESTRICT
#     ON UPDATE CASCADE
# );
import findspark
findspark.init()
from pyspark.sql import SparkSession
import fastf1
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType
from functools import reduce

fastf1.Cache.enable_cache('/home/michal/fastf1_cache')

spark = SparkSession.builder \
        .appName("F1_Telemetry_Processing") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
        .getOrCreate()

url = "jdbc:postgresql://localhost:5432/telemetry"
properties = {"user": "michal", "password": "root", "driver": "org.postgresql.Driver"}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/01 11:26:32 WARN Utils: Your hostname, michal-MS-7996, resolves to a loopback address: 127.0.1.1; using 192.168.0.58 instead (on interface enp2s0)
26/04/01 11:26:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/michal/.ivy2.5.2/cache
The jars for the packages stored in: /home/michal/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-68d57c55-ccd7-43ae-9464-9404458754e4;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.2 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 261ms :: artifacts dl 6ms
	:: modules in use:
	org.checkerfram

In [2]:
dim_session = spark.read.jdbc(url=url, table='dim_session', properties=properties)
dim_session_df = dim_session.collect()
dim_season = spark.read.jdbc(url=url, table='dim_season', properties=properties)
dim_season_df = dim_season.collect()
dim_event = spark.read.jdbc(url=url, table='dim_event', properties=properties)
dim_event_df = dim_event.collect()

season_map = {row['season_id']: row['year'] for row in dim_season_df}

event_map = {
    row["event_id"]: (row["season_id"], row["round_number"])
    for row in dim_event.collect()
}


In [ ]:
loaded = 0
errors = 0
for row in dim_session_df:
    season_id, round_number = event_map[row.event_id]
    try:
        session = fastf1.get_session(season_map[season_id], round_number, row.session_type)
        session.load()
    except:
        print(f"Faild to load for {(season_map[season_id], round_number, row.session_type)}")
        errors += 1
        continue
    weather_df = spark.createDataFrame(session.weather_data)
    weather_df = weather_df.withColumn("session_id", F.lit(row.session_id))
    weather_df = weather_df.withColumn("Time", F.col("Time").cast("long"))
    weather_df = weather_df.select(['session_id', 
                                    'Time', 
                                    'AirTemp', 
                                    'TrackTemp', 
                                    'Humidity', 
                                    'Rainfall', 
                                    'WindDirection', 
                                    'WindSpeed', 
                                    'Pressure'])
    weather_df = weather_df.withColumnsRenamed({'Time': 'timestamp_offset_s', 
                                                'AirTemp': 'air_temp_c', 
                                                'TrackTemp': 'track_temp_c', 
                                                'Humidity': 'humidity_pct', 
                                                'Rainfall': 'rainfall', 
                                                'WindDirection': 'wind_direction', 
                                                'WindSpeed': 'wind_speed_ms', 
                                                'Pressure': 'pressure_mbar'})
    weather_df.write.jdbc(url=url, table='fact_weather', mode='append', properties=properties)
    loaded += 1
print(f'loaded {loaded/(loaded+errors)*100}% ({loaded}, {errors})')

core           INFO 	Loading data for Austrian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO